## Quick Reference: Key Patterns & Code Snippets

### Pattern 1: Circuit Breaker with Polly
```csharp
var circuitBreaker = Policy
    .Handle<HttpRequestException>()
    .CircuitBreakerAsync(
        handledEventsAllowedBeforeBreaking: 5,
        durationOfBreak: TimeSpan.FromSeconds(30),
        onBreak: (outcome, duration) => 
            Console.WriteLine($"Circuit open for {duration.TotalSeconds}s"));
```

### Pattern 2: Transactional Outbox
```csharp
using (var tx = db.BeginTransaction())
{
    // Step 1: Business logic
    order.Status = "APPROVED";
    db.Orders.Update(order);
    
    // Step 2: Write event to outbox (same transaction)
    db.Outbox.Add(new OutboxEvent { 
        EventType = "OrderApproved", 
        Payload = JsonConvert.SerializeObject(order) 
    });
    
    tx.Commit();  // Both or nothing
}
```

### Pattern 3: Idempotent Consumer
```csharp
// Check first
var processed = await dedup.ContainsAsync(messageId);
if (processed) return GetCachedResult(messageId);

// Process
var result = await ProcessEvent(message);

// Store dedup entry
await dedup.StoreAsync(messageId, result, ttl: 30days);
```

### Pattern 4: DI Lifetime Best Practice
```csharp
// WRONG: Scoped in Singleton
builder.Services.AddSingleton<AppConfig>();
builder.Services.AddScoped<DbContext>();

// RIGHT: Let DI handle it
[ApiController]
public class OrderController
{
    [HttpGet]
    public async Task<Order> GetOrder(string id, DbContext db)
    {
        return await db.Orders.FindAsync(id);  // Injected per request
    }
}
```

### Pattern 5: Health Check Implementation
```csharp
public class DatabaseHealthCheck : IHealthCheck
{
    public async Task<HealthCheckResult> CheckHealthAsync(
        HealthCheckContext context, 
        CancellationToken ct = default)
    {
        try
        {
            await _db.Database.ExecuteSqlAsync("SELECT 1", ct);
            return HealthCheckResult.Healthy();
        }
        catch (Exception ex)
        {
            return HealthCheckResult.Unhealthy(exception: ex);
        }
    }
}
```

---

## Real-World Scenarios: Interview Responses

### Scenario 1: "Tell me about your most complex system."
**Answer Template:**
```
System: Healthcare authorization platform
Scale: 100k+ authorizations/month, 5k concurrent users
Tech: C# microservices, Kafka, AKS, Cosmos DB, Azure AD

Architecture:
- Intake service (validation, enrichment)
- Rules engine (80% auto-approve)
- Manual queue (clinician review)
- Notification service (email, SMS, webhooks)

Challenges:
1. Exactly-once processing (idempotency keys, dedup store)
2. Audit trail (event sourcing)
3. Multi-step approval (saga pattern with compensations)
4. External integrations (EHR, insurance, pharmacy)

Solution highlights:
- Circuit breakers prevent cascading failures
- Kafka partitioning by patient ensures order
- 3-tier data (hot Cosmos, warm Blob, cold Archive)
- 99.95% uptime SLA

Impact:
- 40% faster approvals (automated rules)
- 99% accuracy (rules validated)
- Zero data loss (exactly-once semantics)
- Fully auditable (event log)
```

### Scenario 2: "Describe a production outage."
**Answer Template:**
```
Incident: Database connection pool exhausted
Timeline: Friday 4 PM, 15 minute recovery

Root Cause:
- Service leaked connections (forgot to dispose SqlConnection)
- 300 concurrent requests × 5 second timeout = 1500 connections
- Pool size: 100 → exhausted in seconds

Detection:
- Alert: Connection pool queue length > 100
- User impact: 5xx errors on API

Immediate Actions (< 2 mins):
- Paged on-call engineer
- Restarted service (terminated leaked connections)
- Error rate dropped 10% → 0.1%

Root Cause Fix (Saturday):
- Added using() statement around SqlConnection
- Code review caught similar patterns
- Added application insights for connection pool metrics

Prevention:
- Alert if pool queue > 10
- Automated tests for connection leaks
- Architecture review for resource management

Impact:
- Prevented 3 similar incidents in next year
- Established "resource management" design review checklist
```

### Scenario 3: "Difficult stakeholder situation."
**Answer Template:**
```
Situation: VP demanded 15 features in 3 weeks with 3 engineers

My Response:
1. Acknowledged: "Understand urgency, let me show constraints"
2. Presented data:
   - 15 features × 1-2 weeks each = 15-30 weeks
   - Reality: 3 weeks, 3 engineers
   - Math: Can do 2-3 features properly, 5-7 with shortcuts
3. Offered options:
   - Option A: 3 features, quality, on-time
   - Option B: 7 features, risky, delayed, bugs later
   - Option C: 2-3 now, rest in Phase 2 (4 weeks later)
4. Included VP: "Which features matter most?"
5. Ownership: "I'll own delivery, you can shift priorities anytime"

Outcome:
- VP chose option C (3 core features)
- Later phase hit schedule (momentum + team trust)
- Became advocate for realistic planning

Key: Data + Options + Inclusion = Better relationships
```

---

## Study Tips for Success

**Week 1: Foundations**
- Read through Sections 1-5 (C#/.NET deep dive)
- Code exercise: Write async/await state machine, identify allocation hotspots
- Practice: Explain middleware pipeline from memory

**Week 2: Distributed Systems**
- Read Sections 6-9 (events, Kafka, patterns)
- Code exercise: Implement circuit breaker, idempotent consumer
- Practice: Design event-driven system for scenario

**Week 3: Architecture & Production**
- Read Sections 10-12 (Azure, optimization, crisis)
- Design exercise: Complete healthcare system architecture
- Interview practice: Record yourself answering 5 questions

**Week 4: Mock Interviews**
- Have a peer interview you (technical + behavioral)
- Review recordings, identify gaps
- Final polish on stories and confidence

**Interview Day**
- Remember: They want to hear your thinking, not perfect answers
- Pause, think, ask clarifying questions
- Use data to justify decisions
- Show ownership: "I would..." not "We would..."

**You've got this! 💪**

## Section 6-12: Distributed Systems & Production Systems

### Section 6: Event-Driven Architecture (Kafka)
**Interview:** "Design an event-driven healthcare system. How do you handle ordering and duplicates?"

**Key Concepts:**
- Partitioning by patient (ordering guarantee)
- Event sourcing with snapshots
- Idempotent consumer pattern
- Poison message queues
- Saga pattern for distributed transactions

**Quick Answer Template:**
> "I'd partition by patient ID to guarantee ordering. Events go to Kafka topics like `authorization.changes`. Consumers use idempotency keys (stored in dedup database) to prevent duplicates. Failures go to DLQ for manual review. Sagas handle multi-step workflows with compensating transactions for failures."

---

### Section 7: Kafka Consumer Pattern
**Interview:** "Your consumer crashes mid-processing. No data loss, no duplicates?"

**Pattern:** Transactional Outbox + Idempotency
```
1. Begin transaction
2. Update business state
3. Write to dedup store (same transaction)
4. Commit
5. Only then, commit Kafka offset

If crash between steps 3-4:
- Restart, consumer replays message
- Dedup check catches it
- No duplicate
```

---

### Section 8: Polly Resilience Policies
**Interview:** "EHR API is flaky (50% fail). Design resilience."

**Layered Policies:**
```
Timeout(5s) → 
  Retry(3x, exponential) → 
    CircuitBreaker(5 failures, 30s open) → 
      Fallback(cached data) → 
        Result
```

**Key:** Jitter prevents thundering herd, bulkhead prevents cascades

---

### Section 9: Orchestrator vs Choreography
**Interview:** "Healthcare approval workflow: orchestrator or choreography?"

**Answer:**
> "For healthcare, orchestrator. Reason: Audit trail required (regulatory), explicit sequence easier to verify, compensating transactions clear. Trade-off: Tighter coupling, potential bottleneck. At 1000 concurrent, scale horizontally (multiple instances, Redis state)."

---

### Section 10: Azure Architecture
**Interview:** "Design complete healthcare system on Azure."

**Components:**
- Frontend: Static Web App (ReactJS)
- API: APIM + AppGateway
- Compute: AKS (microservices)
- Data: SQL (transactional) + Cosmos DB (events)
- Messaging: Event Hubs + Service Bus
- Auth: Azure AD (clinicians) + B2C (patients)
- Security: KeyVault, Managed Identity, Private endpoints
- Monitoring: Application Insights, Log Analytics

**Cost Calculation:**
- Small (dev): $5-7k/month
- Production: $15-20k/month
- Optimized: $8-10k/month (reserved, spot VMs)

---

### Section 11: Cosmos DB Optimization
**Interview:** "5 years of event data (2B events). Storage optimization?"

**Tiered Strategy:**
```
Hot (6 months): Cosmos DB, RU/s provisioning
Warm (6-12 months): Blob storage, cheap access
Cold (1-5 years): Archive tier, $7.50/month for 1.5TB

Archive Strategy:
- Every 6 months, stream events to Parquet
- Compress (50% compression)
- Delete from Cosmos
- Keep pointer table
- Old queries load from Blob (slower OK for historical)

Cost Reduction:
$28,800/month (all in Cosmos) → 
$469/month (tiered)
= 94% savings!
```

---

### Section 12: Production Crisis Management
**Interview:** "Describe a production incident. What did you do?"

**Incident Protocol:**
```
1. DETECTION (Alert)
   - Error rate spike
   - Latency increase
   - Health check failed

2. MITIGATION (Immediate, < 5 mins)
   - Scale up (add capacity)
   - Reduce load (feature toggle)
   - Restart service
   - Failover to backup

3. ROOT CAUSE (Investigation)
   - Check metrics (CPU, memory, network)
   - Check logs (errors, exceptions)
   - Check dependencies (external APIs)
   - Hypothesis → Test

4. PERMANENT FIX
   - Code change (bug fix)
   - Config change (timeout tuning)
   - Architecture change (circuit breaker)
   - Deploy, verify, monitor

5. POSTMORTEM
   - What happened?
   - Why did it happen?
   - How do we prevent next time?
   - Action items with owners
```

**Real Example:**
> "EHR API was slow, causing timeouts. I immediately reduced timeout from 30s to 5s (mitigation), preventing ThreadPool starvation. Root cause: EHR CPU spike. Permanent fix: Added circuit breaker + cached fallback. Prevented 3 outages in next 3 months."

---

## Summary: Interview Preparation Checklist

**Technical Mastery:**
- [ ] Can explain async/await state machine from IL code
- [ ] Diagnose and fix GC pressure (> 5% CPU)
- [ ] Design clean architecture with proper DI lifetimes
- [ ] Implement resilience patterns (Polly, circuit breaker)
- [ ] Design event-driven system with exactly-once semantics
- [ ] Azure architecture for 100k to 10M users

**Soft Skills:**
- [ ] Tell story of time-critical delivery (3 weeks, tight team)
- [ ] Crisis management (production outage, incident response)
- [ ] Difficult decision (saying no to stakeholder)
- [ ] Technical leadership (mentoring, design reviews)

**Practice:**
- [ ] Time yourself: 10 min answers for each section
- [ ] Whiteboard: Draw architectures from memory
- [ ] Code: Write resilience patterns without reference
- [ ] Mock interview: Record yourself, review

**Success Metrics:**
- Interview: 70+ points (out of 100)
- Confidence: Calm under pressure
- Leadership: Balance speed with quality

## Section 5: Advanced API Architecture Patterns

### Interview Question
**"Design a clean architecture for a healthcare API with proper DI lifetimes. What's a captive dependency?"**

### Clean Architecture Layers
```csharp
// Layer 1: Controllers (Entry point)
[ApiController]
[Route("api/[controller]")]
public class AuthorizationsController : ControllerBase
{
    private readonly IAuthorizationService _service;
    
    public AuthorizationsController(IAuthorizationService service)
    {
        _service = service;  // Injected
    }
    
    [HttpPost]
    public async Task<CreateAuthorizationResponse> Create(
        CreateAuthorizationRequest request)
    {
        var authorization = await _service.CreateAsync(request);
        return new CreateAuthorizationResponse(authorization);
    }
}

// Layer 2: Services (Business logic)
public class AuthorizationService : IAuthorizationService
{
    private readonly IAuthorizationRepository _repository;
    private readonly IEligibilityService _eligibility;
    private readonly IKafkaProducer _events;
    
    public async Task<Authorization> CreateAsync(CreateAuthorizationRequest request)
    {
        // Validate
        if (!await _eligibility.CheckAsync(request.PatientId))
            throw new InvalidOperationException("Patient ineligible");
        
        // Create
        var auth = new Authorization(request);
        
        // Persist
        await _repository.AddAsync(auth);
        await _repository.SaveChangesAsync();
        
        // Publish event
        await _events.PublishAsync(new AuthorizationCreatedEvent(auth));
        
        return auth;
    }
}

// Layer 3: Repositories (Data access)
public class AuthorizationRepository : IAuthorizationRepository
{
    private readonly DbContext _db;
    
    public async Task<Authorization> GetAsync(string id)
    {
        return await _db.Authorizations.FindAsync(id);
    }
    
    public async Task AddAsync(Authorization auth)
    {
        _db.Authorizations.Add(auth);
    }
    
    public async Task SaveChangesAsync()
    {
        await _db.SaveChangesAsync();
    }
}

// Layer 4: Data Access (Entity Framework)
public class DbContext : Microsoft.EntityFrameworkCore.DbContext
{
    public DbSet<Authorization> Authorizations { get; set; }
}
```

### DI Lifetime Management
```csharp
// Program.cs
builder.Services
    // TRANSIENT: New instance every time
    .AddTransient<IAuthorizationValidator, AuthorizationValidator>()
    
    // SCOPED: One per request (HTTP scope)
    .AddScoped<IAuthorizationService, AuthorizationService>()
    .AddScoped<IAuthorizationRepository, AuthorizationRepository>()
    .AddScoped<DbContext>()
    
    // SINGLETON: One for entire app lifetime
    .AddSingleton<IMemoryCache>(new MemoryCache(new MemoryCacheOptions()))
    .AddSingleton<IHealthCheck, SystemHealthCheck>();

// HttpClient best practice: AddHttpClient
builder.Services
    .AddHttpClient<IEligibilityService, EligibilityService>()
    .ConfigureHttpClient(client => 
    {
        client.BaseAddress = new Uri("https://eligibility-api.internal");
        client.DefaultRequestHeaders.Add("Authorization", "Bearer token");
    });
```

### Captive Dependency Anti-Pattern
```csharp
// ❌ WRONG: Scoped service in Singleton
builder.Services.AddSingleton<AppConfig>();  // Singleton

builder.Services.AddScoped<DbContext>();  // Scoped

// This is WRONG:
public class AppConfig  // Singleton
{
    private readonly DbContext _db;  // Tries to use Scoped service
    
    public AppConfig(DbContext db)
    {
        _db = db;  // ← Captive dependency!
    }
}
// Problem: DbContext stored in Singleton, reused across all requests
// Result: Database connection shared, connection pooling broken, race conditions

// ✓ CORRECT: Pass through request
public class AuthorizationController
{
    [HttpGet("{id}")]
    public async Task<Authorization> Get(string id, DbContext db)
    {
        return await db.Authorizations.FindAsync(id);
    }
}
// DbContext injected per request, scoped correctly
```

### API Versioning Strategies
```csharp
// Strategy 1: URL Path
// GET /api/v1/authorizations
// GET /api/v2/authorizations

builder.Services.AddApiVersioning(options =>
{
    options.AssumeDefaultVersionWhenUnspecified = true;
    options.DefaultApiVersion = new ApiVersion(1, 0);
    options.ReportApiVersions = true;
});

[ApiController]
[Route("api/v{version:apiVersion}/[controller]")]
public class AuthorizationsController : ControllerBase
{
    [HttpGet("{id}")]
    [MapToApiVersion("1.0")]
    public async Task<AuthorizationV1> GetV1(string id) { }
    
    [HttpGet("{id}")]
    [MapToApiVersion("2.0")]
    public async Task<AuthorizationV2> GetV2(string id) { }
}

// Strategy 2: Header
// GET /api/authorizations
// Header: api-version: 2.0

// Strategy 3: Query Parameter
// GET /api/authorizations?api-version=2.0
```

### Idempotent API Design
```csharp
// Requirement: Creating same authorization twice = Same result
public class CreateAuthorizationRequest
{
    public string IdempotencyKey { get; set; }  // Client-provided
    public string PatientId { get; set; }
    public string InsuranceId { get; set; }
}

public async Task<CreateAuthorizationResponse> Create(
    CreateAuthorizationRequest request)
{
    // Check if already created
    var existing = await _cache.GetAsync($"idempotency:{request.IdempotencyKey}");
    if (existing != null)
    {
        return existing;  // Return cached result
    }
    
    // Create new
    var auth = new Authorization(request);
    await _repository.AddAsync(auth);
    await _repository.SaveChangesAsync();
    
    // Cache result (TTL 24 hours)
    await _cache.SetAsync(
        $"idempotency:{request.IdempotencyKey}",
        auth,
        new DistributedCacheEntryOptions 
        { 
            AbsoluteExpirationRelativeToNow = TimeSpan.FromHours(24) 
        });
    
    return auth;
}

// Client usage:
POST /api/authorizations
{
    "idempotencyKey": "client-generated-uuid",
    "patientId": "PAT-123",
    "insuranceId": "INS-456"
}

// If network fails, client retries with same UUID
// Server returns cached result (no duplicate created)
```

## Section 4: ASP.NET Core Request Pipeline & Middleware

### Interview Question
**"Explain the complete request flow from Kestrel to response. Where should business logic go?"**

### The Pipeline (In Order)
```
Client Request
    ↓
Kestrel (HTTP server)
    ↓
[Middleware 1] [Middleware 2] ... [Middleware N]
    ↓
Routing
    ↓
Authorization & Authentication Filters
    ↓
Resource Filters (caching, compression)
    ↓
Action Filters (before action)
    ↓
Controller Action
    ↓
Action Filters (after action)
    ↓
Result Filters
    ↓
Exception Filters
    ↓
Middleware (reverse order on response)
    ↓
Kestrel
    ↓
Client Response
```

### Middleware Ordering (Critical)
```csharp
// app.Program.cs - Order matters!
var app = builder.Build();

// 1. Logging (early, captures all requests)
app.UseWhen(context => context.Request.Path.StartsWithSegments("/api"),
    appBuilder => appBuilder.UseMiddleware<RequestLoggingMiddleware>());

// 2. Exception handling (catches from all downstream)
app.UseExceptionHandler("/error");

// 3. HTTPS redirect
app.UseHttpsRedirection();

// 4. Security headers
app.Use(async (context, next) =>
{
    context.Response.Headers.Add("X-Frame-Options", "DENY");
    context.Response.Headers.Add("X-Content-Type-Options", "nosniff");
    await next();
});

// 5. CORS (must be before auth)
app.UseCors("AllowHealthcareUI");

// 6. Authentication
app.UseAuthentication();

// 7. Authorization
app.UseAuthorization();

// 8. Custom middleware
app.UseMiddleware<JwtValidationMiddleware>();

// 9. API versioning
app.UseApiVersioning();

// 10. Routing (must be second-to-last)
app.UseRouting();

// 11. Endpoints (must be last)
app.MapControllers();
app.MapHealthChecks("/health");
```

### Filter Execution Order
```
For GET /api/authorizations/{id}

1. Authorization Filter
   - Check [Authorize], roles, policies
   - If fails: Return 403

2. Resource Filter (OnResourceExecuting)
   - Check cache, skip action if found
   - Example: [ResponseCache(Duration=300)]

3. Action Filter (OnActionExecuting)
   - Modify request, validate input
   - Example: Custom validation

4. ACTION EXECUTES
   public async Task<Authorization> Get(string id)
   {
       return await _db.Authorizations.FindAsync(id);
   }

5. Action Filter (OnActionExecuted)
   - Modify response
   - Example: Add metadata

6. Result Filter
   - Modify response body
   - Example: Wrap in API response

7. Exception Filter (if exception thrown)
   - Handle errors, log
   - Example: Convert exceptions to HTTP status
```

### Where Does Business Logic Go?
```
❌ NOT IN MIDDLEWARE:
- Database queries
- Business rules
- Complex workflows
- (Middleware is for cross-cutting concerns)

✓ IN CONTROLLERS:
- Parse input
- Call service
- Return response

✓ IN SERVICES:
- Business logic
- Database queries
- Validation
- Transactions

✓ IN REPOSITORIES:
- Database access
- Queries only
- No business logic

✓ IN FILTERS:
- Authorization checks
- Caching
- Logging
- Request/response transformation
```

### Global Error Handling Pattern
```csharp
// Centralized exception handler (middleware)
public class ExceptionHandlingMiddleware
{
    private readonly RequestDelegate _next;
    
    public async Task InvokeAsync(HttpContext context)
    {
        try
        {
            await _next(context);
        }
        catch (UnauthorizedAccessException ex)
        {
            context.Response.StatusCode = 401;
            await context.Response.WriteAsJsonAsync(new { error = "Unauthorized" });
        }
        catch (InvalidOperationException ex)
        {
            context.Response.StatusCode = 400;
            await context.Response.WriteAsJsonAsync(new { error = ex.Message });
        }
        catch (Exception ex)
        {
            _logger.LogError(ex, "Unhandled exception");
            context.Response.StatusCode = 500;
            await context.Response.WriteAsJsonAsync(new { error = "Internal server error" });
        }
    }
}

// Register in Program.cs
app.UseMiddleware<ExceptionHandlingMiddleware>();
```

### Health Checks
```csharp
// Program.cs
builder.Services.AddHealthChecks()
    .AddSqlServer(configuration.GetConnectionString("DefaultConnection"))
    .AddCheck<KafkaHealthCheck>("Kafka")
    .AddCheck<AzureKeyVaultHealthCheck>("KeyVault");

app.MapHealthChecks("/health", new HealthCheckOptions
{
    ResponseWriter = WriteHealthCheckResponse
});

// Output:
GET /health
{
    "status": "Healthy",
    "checks": {
        "SqlServer": "Healthy",
        "Kafka": "Healthy",
        "KeyVault": "Healthy"
    },
    "totalDuration": "00:00:00.1234567"
}
```

## Section 3: Garbage Collection & Memory Management

### Interview Question
**"Your ASP.NET Core API has 30% CPU overhead from GC. How would you diagnose and fix this?"**

### GC Generations Deep Dive
```
Generation 0 (Ephemeral):
- Collected frequently (every ~64KB allocation)
- Fast collection (10-50ms)
- Default destination for all new objects
- Problem: Allocate-heavy code hits Gen0 constantly

Generation 1 (Ephemeral):
- Promoted from Gen0 if survives 1 collection
- Collected less frequently
- Smaller size than Gen0
- Transient objects live here briefly

Generation 2 (Full Heap):
- Promoted from Gen1 if survives 1 collection
- Collected infrequently (~30s in healthy system)
- Full GC blocks all threads (stop-the-world)
- PROBLEM: 30% CPU usually = frequent Gen2 collections
```

### Large Object Heap (LOH)
```
LOH characteristics:
- Objects >= 85KB go to LOH
- Never compacted (fragmentation risk)
- Full GC collection when LOH pressure high
- Example: DataTable (often 100KB+) → LOH

Problem Scenario:
- Create DataTable(1MB) → LOH
- Query 1000 rows × 1000 queries = 1GB LOH pressure
- Gen2 full collections trigger
- 30% CPU overhead

Solution:
- Use smaller objects (Stream, ArrayPool<T>)
- Reuse objects (object pool for DataTable)
- Use Span<T> (stack allocation, no GC)
```

### Diagnostic Metrics
```
Healthy System:
- Gen0 collections/sec: < 100
- Gen1 collections/sec: < 10
- Gen2 collections/sec: < 1
- LOH allocations/sec: < 100

Problematic System:
- Gen0: > 1000 (thrashing)
- Gen1: > 100 (objects not dying fast)
- Gen2: > 10 (frequent full GC, BAD)
- LOH: > 1000 (large object pressure)
```

### Profiling Tools & Techniques
```csharp
// Using dotnet-counters to monitor live
var counters = new PerformanceCounter(
    "\.NET CLR Memory",
    "Allocated Bytes/sec");

// Using EventPipe to capture GC events
dotnet trace collect --process-id <PID> --profile gc-verbose

// In-code allocation tracking (Benchmark.NET)
var stopwatch = Stopwatch.StartNew();
var allocation = GC.GetTotalMemory(true);
DoWork();
var allocatedBytes = GC.GetTotalMemory(false) - allocation;
stopwatch.Stop();

Console.WriteLine($"Allocated: {allocatedBytes} bytes in {stopwatch.ElapsedMilliseconds}ms");
```

### Common Allocations to Avoid
| Problem | Cost | Solution |
|---------|------|----------|
| String concat | 10x allocation | Use StringBuilder |
| LINQ chains | Per object | Use IEnumerable.SingleOrDefault() |
| Array resizing | Frequent allocations | Use List<T> with capacity |
| Struct boxing | Gen0 allocation | Use generic constraints |
| async/await | Small allocation | Only where needed |
| exception catch | Gen2 allocation | Don't use for control flow |

### Real-World Fix Example
```csharp
// BEFORE: 30% CPU from GC
public IEnumerable<Authorization> FilterAuthorizations(List<Authorization> items)
{
    return items
        .Where(x => x.Status == "PENDING")
        .OrderBy(x => x.CreatedAt)
        .Select(x => new { x.Id, x.PatientId })
        .ToList();  // ← ALLOCATES: List, select objects
}

// AFTER: <5% CPU
public ArrayPool<Authorization> FilterAuthorizations(
    List<Authorization> items,
    List<Authorization> results)
{
    results.Clear();
    foreach (var item in items)
    {
        if (item.Status == "PENDING")
        {
            results.Add(item);  // ← NO ALLOCATION
        }
    }
    // Sort in-place (no allocation)
    results.Sort((a, b) => a.CreatedAt.CompareTo(b.CreatedAt));
    return results;
}
```

### Key Interview Takeaway
> *"High GC overhead usually means high allocation rate. Profile with dotnet-counters, find allocation hotspots, eliminate unnecessary object creation using spans, pooling, or structural design changes."*

## Section 2: Async/Await State Machine Internals

### Interview Question
**"Decompile an async method and explain how it works at IL level. Why is ConfigureAwait(false) important?"**

### The Pattern: Behind the Scenes
```csharp
// What you write:
public async Task<EligibilityResponse> CheckEligibilityAsync(string patientId)
{
    var response = await _httpClient.GetAsync($"/api/eligibility/{patientId}");
    var json = await response.Content.ReadAsStringAsync();
    return JsonConvert.DeserializeObject<EligibilityResponse>(json);
}

// What compiler generates:
public class CheckEligibilityAsync_StateMachine : IAsyncStateMachine
{
    public int State;
    public AsyncTaskMethodBuilder<EligibilityResponse> Builder;
    
    public void MoveNext()
    {
        try
        {
            if (State == 0)  // Initial state
            {
                // Await httpClient.GetAsync()
                var awaiter = _httpClient.GetAsync(...).GetAwaiter();
                if (!awaiter.IsCompleted)
                {
                    State = 1;
                    Builder.AwaitUnsafeOnCompleted(ref awaiter, ref this);
                    return;
                }
                State = -1;
                var response = awaiter.GetResult();
            }
            
            if (State == 1)  // After first await
            {
                // Await response.Content.ReadAsStringAsync()
                var awaiter = response.Content.ReadAsStringAsync().GetAwaiter();
                if (!awaiter.IsCompleted)
                {
                    State = 2;
                    Builder.AwaitUnsafeOnCompleted(ref awaiter, ref this);
                    return;
                }
                State = -1;
                var json = awaiter.GetResult();
                var result = JsonConvert.DeserializeObject<EligibilityResponse>(json);
                Builder.SetResult(result);
            }
        }
        catch (Exception ex)
        {
            State = -2;
            Builder.SetException(ex);
        }
    }
}
```

### Key Insights
1. **MoveNext() pattern**: Each await becomes a state transition
2. **IsCompleted optimization**: If already complete, no allocation
3. **SynchronizationContext**: By default, result marshalled back (UI thread problem)
4. **ConfigureAwait(false)**: Prevents SynchronizationContext capture (library best practice)

### Real Scenario
```
Problem: UI thread awaits network call
- Network completes on ThreadPool
- Awaiter tries to marshal back to UI thread
- UI thread blocked doing other work
- Deadlock: UI waiting for result, ThreadPool waiting for UI

Solution: Use ConfigureAwait(false) in libraries
- Network completes on ThreadPool
- No attempt to marshal back
- Continues on ThreadPool thread (safe)
- UI thread never blocked
```

### Performance Characteristics
- **Allocation cost**: 0 if already completed (IsCompleted=true)
- **Context switching cost**: ~100μs per marshal (ConfigureAwait impact)
- **ThreadPool pressure**: High sync-over-async usage = starvation

## Section 1: C# & .NET Internals Deep Dive

### Core Concepts
- **async/await state machines**: Compilation to MoveNext(), TaskAwaiter pattern
- **Task vs Thread vs ValueTask**: When to use each, performance implications
- **CPU-bound vs IO-bound**: Threadpool implications, ConfigureAwait(false)
- **ThreadPool starvation**: Recognizing and preventing
- **Sync-over-async deadlocks**: Classic pattern that breaks UI
- **Garbage collection generations**: Gen0/Gen1/Gen2, LOH behavior
- **Struct vs class trade-offs**: Allocation, boxing/unboxing costs
- **Memory profiling**: Identifying leaks in production

### Key Interview Questions
1. **Explain async/await state machines**: What happens when you await?
2. **ValueTask vs Task**: When would you use ValueTask and why?
3. **What causes ThreadPool starvation?**: Real scenario examples
4. **Sync-over-async**: Why is `task.Result` dangerous on UI thread?
5. **GC pressure**: How would you diagnose high allocation rates?
6. **Struct pitfalls**: When would packing structs harm performance?
7. **Dependency injection lifetimes**: What's a captive dependency?
8. **ConfigureAwait(false)**: Why is this critical in libraries?

### Performance Metrics to Know
- **Allocation rate**: > 1GB/sec = problematic
- **Gen2 collection frequency**: Should be < 1 per minute in healthy system
- **ThreadPool queue length**: If growing, starvation likely
- **Await latency**: < 1ms for scalability

# Senior Technical Leadership Interview Preparation
## Comprehensive Deep Dive: C#/.NET, Distributed Systems & Azure Architecture

**Target Roles:** Senior Full-Stack Architect, Technical Lead, Associate Solution Architect  
**Timeline:** 3-4 week intensive preparation  
**Focus:** Technical mastery + leadership capability  

### Notebook Structure
This notebook covers 12 major technical domains with practical examples, architectural patterns, and production-ready solutions for healthcare and fintech systems.

**Time Estimate:** 15-20 hours active study  
**Prerequisites:** 5+ years experience with C#/.NET, cloud architecture, distributed systems